# GPT-SoVITS — Google Colab Final Notebook

Notebook final dari setup GPT-SoVITS yang berhasil digunakan untuk **zero-shot anime-character voice cloning** di Google Colab dengan GPU.

## Final environment

- Repository: `RVC-Boss/GPT-SoVITS`
- Conda environment: `GPTSoVits`
- Python: 3.10.x
- PyTorch: `2.10.0+cu126`
- TorchAudio: `2.10.0+cu126`
- TorchCodec: `0.10.0`
- Gradio: `4.44.1`
- FastAPI: `0.115.2`
- Pydantic: `2.9.2`
- CUDA build: 12.6

## Workflow

1. Clone repository
2. Install Miniconda
3. Accept Conda ToS
4. Create `GPTSoVits`
5. Install GPT-SoVITS
6. Fix dependency versions
7. Verify pretrained models
8. Start Main WebUI
9. Start TTS Inference WebUI
10. Perform zero-shot voice cloning


## 1. Clone GPT-SoVITS repository


In [ ]:
%cd /content
!git clone https://github.com/RVC-Boss/GPT-SoVITS.git
%cd /content/GPT-SoVITS


## 2. Install Miniconda


In [ ]:
%cd /content
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -p /content/miniconda
!/content/miniconda/bin/conda init bash
!/content/miniconda/bin/conda update -n base -c defaults conda -y


## 3. Accept Conda Terms of Service


In [ ]:
%%bash
source /content/miniconda/etc/profile.d/conda.sh
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r


## 4. Create Conda environment


In [ ]:
%%bash
source /content/miniconda/etc/profile.d/conda.sh
conda create -n GPTSoVits python=3.10 -y
conda activate GPTSoVits

echo "Python:"
python --version
echo "Python path:"
which python


## 5. Install GPT-SoVITS


In [ ]:
%%bash
source /content/miniconda/etc/profile.d/conda.sh
conda activate GPTSoVits
cd /content/GPT-SoVITS

bash install.sh --device CU126 --source HF


## 6. Fix dependency versions

Dependency berikut adalah kombinasi final yang berhasil digunakan pada runtime ini.


In [ ]:
%%bash
source /content/miniconda/etc/profile.d/conda.sh
conda activate GPTSoVits

# Gradio compatibility
pip install --force-reinstall "fastapi==0.115.2"

# TTS WebUI schema compatibility
pip install --force-reinstall "pydantic==2.9.2" "pydantic-core==2.23.4"

# Matching PyTorch + TorchAudio for CUDA 12.6
pip install --force-reinstall \
    "torch==2.10.0" \
    "torchaudio==2.10.0" \
    --index-url https://download.pytorch.org/whl/cu126


## 7. Fix TorchCodec

Pada proses troubleshooting, `torchcodec 0.17.0+cu126` menghasilkan:

```text
undefined symbol: torch_from_blob
```

Environment final yang berhasil menggunakan `torchcodec==0.10.0` dengan PyTorch `2.10.0+cu126`.


In [ ]:
%%bash
source /content/miniconda/etc/profile.d/conda.sh
conda activate GPTSoVits

pip uninstall -y torchcodec
pip install "torchcodec==0.10.0" \
    --index-url https://download.pytorch.org/whl/cu126


## 8. Verify final environment


In [ ]:
%%bash
source /content/miniconda/etc/profile.d/conda.sh
conda activate GPTSoVits

python - <<'PY'
import torch
import torchaudio
import torchcodec
import gradio
import fastapi
import pydantic

print("torch      :", torch.__version__)
print("torchaudio :", torchaudio.__version__)
print("torchcodec :", torchcodec.__version__)
print("gradio     :", gradio.__version__)
print("fastapi    :", fastapi.__version__)
print("pydantic   :", pydantic.__version__)
print("CUDA       :", torch.version.cuda)
print("GPU ready  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU        :", torch.cuda.get_device_name(0))
PY


## 9. Check pretrained models


In [ ]:
!find /content/GPT-SoVITS/GPT_SoVITS/pretrained_models -maxdepth 2 -type f | sort | head -100


## 10. Important runtime setting

Gunakan:

```bash
export MPLBACKEND=Agg
```

untuk menghindari error Matplotlib backend yang muncul pada runtime Colab.

Gunakan:

```bash
export is_share=True
```

agar Gradio membuat public sharing URL.

`inference_webui.py` sudah memiliki `share=is_share`, jadi source code tidak perlu dimodifikasi.


## 11. Start Main GPT-SoVITS WebUI


In [ ]:
%%bash
source /content/miniconda/etc/profile.d/conda.sh
conda activate GPTSoVits
cd /content/GPT-SoVITS

export MPLBACKEND=Agg
export is_share=True

python -u webui.py


## 12. Start TTS Inference WebUI


In [ ]:
%%bash
source /content/miniconda/etc/profile.d/conda.sh
conda activate GPTSoVits
cd /content/GPT-SoVITS

export MPLBACKEND=Agg
export is_share=True

nohup python -u GPT_SoVITS/inference_webui.py Auto \
    > /content/tts_webui.log 2>&1 &

echo "PID: $!


### Check TTS WebUI log


In [ ]:
!sleep 5
!tail -40 /content/tts_webui.log


### Check port 9872


In [ ]:
!ss -ltnp | grep 9872 || true


## 13. Zero-shot voice cloning

Untuk zero-shot, gunakan:

- Reference audio: sekitar 3–10 detik
- Satu speaker
- Audio bersih
- Minim noise/music
- Reference text harus **persis** sesuai ucapan pada reference audio
- Target text adalah kalimat baru
- Reference language: Japanese (`ja`)
- Target language: Japanese (`ja`)

### Example yang berhasil digunakan

**Reference text**

```text
宇宙に秘められた究極の神秘も解き明かせてないのに、生活のことなんて気にしてる暇があると思う？
```

**Target text**

```text
こんにちは、お元気ですか？
```

Log menunjukkan target text berhasil melewati preprocessing dan T2S decoding sebelum audio reference dibaca.

Untuk zero-shot, **tidak perlu** melakukan dataset formatting atau fine-tuning terlebih dahulu.


## 14. Troubleshooting record

### `Conda Not Found`

Solusi: install Miniconda di `/content/miniconda`.

### Gradio/Jinja error

Solusi:

```bash
pip install --force-reinstall "fastapi==0.115.2"
```

### Matplotlib backend error

Solusi:

```bash
export MPLBACKEND=Agg
```

### `libcudart.so.13` / TorchAudio loading error

Solusi: gunakan PyTorch dan TorchAudio yang matching:

```text
torch       2.10.0+cu126
torchaudio  2.10.0+cu126
```

### TorchCodec `undefined symbol: torch_from_blob`

Penyebab: `torchcodec 0.17.0+cu126` tidak cocok dengan PyTorch `2.10.0+cu126` pada environment ini.

Solusi:

```bash
pip uninstall -y torchcodec
pip install "torchcodec==0.10.0" --index-url https://download.pytorch.org/whl/cu126
```

Setelah mengganti dependency, proses Python TTS harus direstart.


## 15. Runtime notes

Google Colab runtime bersifat sementara. Jika runtime di-reset, repository, environment, dan model yang berada di `/content` perlu disiapkan kembali.

Untuk penggunaan publik atau komersial, pastikan memiliki hak penggunaan yang sesuai atas reference voice, rekaman, dan performa suara yang digunakan.
